In [ ]:
import numpy as np
import ufl

from mpi4py import MPI

import dolfinx
import dolfinx.fem.petsc
import dolfinx.mesh
import basix.ufl
from petsc4py import PETSc

N = 2

domain = dolfinx.mesh.create_unit_square(
    MPI.COMM_WORLD, 
    N, N, 
    dolfinx.mesh.CellType.quadrilateral
)
coords = domain.geometry.x
xi = coords[:, 0]   # x-coordinates of the unit square
eta = coords[:, 1]  # y-coordinates of the unit square
x_new = 4.8 * xi
y_new = 4.4 * eta + 4.4 * xi - 2.8 * xi * eta
domain.geometry.x[:, 0] = x_new
domain.geometry.x[:, 1] = y_new


dim = domain.topology.dim
print(f"Mesh topology dimension d={dim}.")

degree = 2
#shape = (dim,)  # this means we want a vector field of size `dim`
v_elem = basix.ufl.element(
    "Lagrange", 
    domain.topology.cell_name(), 
    degree, 
    shape=(dim,)
)
V = dolfinx.fem.functionspace(domain, v_elem)

u_sol = dolfinx.fem.Function(V, name="Displacement")

E = dolfinx.fem.Constant(domain, 70.)
nu = dolfinx.fem.Constant(domain, 1./3.)

lmbda = E * nu / (1. + nu) / (1. - 2. * nu)
mu = E / 2. / (1. + nu)

def right_boundary(x):
    return np.isclose(x[0], 4.8)
def left_boundary(x):
    return np.isclose(x[0], 0.)

facet_dim = domain.topology.dim-1
boundary_facets = dolfinx.mesh.locate_entities_boundary(domain, facet_dim, right_boundary)
boundary_tags = dolfinx.mesh.meshtags(
    domain,
    facet_dim,
    boundary_facets,
    np.full(len(boundary_facets), 1, dtype=np.int32)
)

def epsilon(v):
    return ufl.sym(ufl.grad(v))


def sigma(v):
    return lmbda * ufl.tr(epsilon(v)) * ufl.Identity(dim) + 2. * mu * epsilon(v)

u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

T = dolfinx.fem.Constant(
    domain, 
    dolfinx.default_scalar_type((0.0, 6.25))
)


custom_metadata = {"quadrature_degree": 8}
custom_dx = ufl.Measure("dx", domain=domain, metadata=custom_metadata)
custom_ds = ufl.Measure("ds", domain=domain, subdomain_data=boundary_tags, metadata=custom_metadata)
a = ufl.inner(sigma(u), epsilon(v)) * custom_dx
L = ufl.inner(T, v)*custom_ds(1)

left_dofs = dolfinx.fem.locate_dofs_geometrical(V, left_boundary)
zero_vec = np.zeros(dim, dtype=dolfinx.default_scalar_type)
bcs = [
    dolfinx.fem.dirichletbc(zero_vec, left_dofs, V),
]
problem = dolfinx.fem.petsc.LinearProblem(
    a, L, u=u_sol, bcs=bcs,
    petsc_options_prefix="cooks_membrane",
    petsc_options={
        "ksp_type": "preonly", 
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps"}
)
problem.solve()
# A = dolfinx.fem.petsc.assemble_matrix(a, bcs=bcs)
# A.assemble()

# # Assemble b
# b = dolfinx.fem.petsc.assemble_vector(L)
# dolfinx.fem.petsc.apply_lifting(b, [a], bcs=[bcs])
# b.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES,
#               mode=PETSc.ScatterMode.REVERSE)
# dolfinx.fem.petsc.set_bc(b, bcs)
#b_lagrange = problem.b
#A_lagrange = problem.A

In [ ]:
import pyvista
from dolfinx.plot import vtk_mesh

topology, cell_types, geometry = vtk_mesh(V)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# The solution array is 1D. We reshape it to N x 2 (since dim=2)
u_values = u_sol.x.array.reshape(-1, dim)

# PyVista requires 3D vectors to warp the mesh. We pad the 2D vectors with Z=0.
u_3d = np.zeros((u_values.shape[0], 3), dtype=np.float64)
u_3d[:, :dim] = u_values

# Attach the 3D displacement vectors to the grid
grid.point_data["Displacement"] = u_3d
grid.set_active_vectors("Displacement")

# Warp the grid by the displacement. 
warp_factor = 1.0
warped_grid = grid.warp_by_vector("Displacement", factor=warp_factor)

# Plotting
plotter = pyvista.Plotter()
plotter.add_text(f"Deformed Mesh (magnified {warp_factor}x)", font_size=14)

# Show the original undeformed mesh as a wireframe
plotter.add_mesh(grid, style="wireframe", color="black", opacity=0.3, label="Undeformed")

# Show the deformed mesh
plotter.add_mesh(warped_grid, show_edges=False, scalars="Displacement", cmap="coolwarm", label="Deformed")

plotter.view_xy()  # Set camera to view the X-Y plane directly
plotter.show(jupyter_backend="static")

In [ ]:
print(f"Top right corner vertical displacement: {np.max(u_3d[:,1])*10:.2f}mm")

# B-Splines

In [ ]:
from thbsplines.hierarchical_space import HierarchicalSpace
import numpy as np
import dolfinx
from mpi4py import MPI
import basix.ufl
import pyvista

from dolfinx import default_real_type, default_scalar_type
rtype = default_real_type
dtype = default_scalar_type
import ufl
from copy import deepcopy
from scipy.integrate import quad

from thbsplines.refinement import refine
from thbsplines.fenicsx.mesh import build_mesh, FastMidpointMapper
from thbsplines.fenicsx.functionspace import build_dofmap, fill_function_space, create_spline_space
from thbsplines.fenicsx.solvers import solve_problem_vector_field, enforce_dirichlet_boundary
from thbsplines.fenicsx.postprocessing import map_spline_to_legendre
from thbsplines.fenicsx.kernels import make_vector_bilinear_kernel, make_vector_linear_kernel
from thbsplines.fenicsx.adaptivity import dorfler_marking

In [ ]:
n_refinements = 1
p0 = 2
knotsx = np.array([0,0,0.5,1,1], dtype=np.float64)
knotsx = refine(knotsx, p=p0, n_times=n_refinements)
log_initial_mesh_size = np.log2(np.max(np.diff(knotsx)))
knotsy = np.array([0,0,0.5,1,1], dtype=np.float64)
knotsy = refine(knotsy, p0, n_times=n_refinements)
err_cells = {}
T_err_cells = {}
hs = HierarchicalSpace(knots=[knotsx, knotsy], degrees=[p0])
p0_dual = p0+1
m=3
knotsx_dual = refine(knotsx, p0_dual, 0)
knotsy_dual = refine(knotsy, p0_dual, 0)
hs_dual = HierarchicalSpace([knotsx_dual, knotsy_dual], [p0_dual])

In [ ]:
def my_dict_update(a,b):
    for level in a.keys():
        if level in b:
            b[level] = np.concatenate((a[level], b[level]))
        else:
            b[level] = a[level]
    return b

In [ ]:
# mesh_data = hs.hmesh.aelem_level
# h0 = 0.125          # Base side-length at Level 0
# domain_width = 1.0 # Total width of your domain
# N0 = int(domain_width / h0) # Number of base cells per side (4 in this case)

# with open("mesh_data_p3.dat", "w") as f_write:
#     # Header for PGFPlots
#     f_write.write("x y\n")
    
#     for level, indices in mesh_data.items():
#         # Number of cells along an axis at the current level
#         NL = N0 * (2 ** level)
#         # Element side-length at the current level
#         hL = h0 / (2 ** level)
        
#         for idx in indices:
#             # Lexicographical decoding to grid coordinates
#             x_idx = idx // NL
#             y_idx = idx % NL
            
#             # If your convention has X changing slowest, swap the lines above to:
#             # x_idx = idx // NL
#             # y_idx = idx % NL
            
#             # Calculate physical coordinate boundaries
#             xmin = x_idx * hL
#             ymin = y_idx * hL
#             xmax = xmin + hL
#             ymax = ymin + hL
            
#             # Write out the 5 vertices to close the square path
#             f_write.write(f"{xmin:.6f} {ymin:.6f}\n")
#             f_write.write(f"{xmax:.6f} {ymin:.6f}\n")
#             f_write.write(f"{xmax:.6f} {ymax:.6f}\n")
#             f_write.write(f"{xmin:.6f} {ymax:.6f}\n")
#             f_write.write(f"{xmin:.6f} {ymin:.6f}\n")
#             f_write.write("\n") # CRITICAL: Tells PGFPlots to lift the pen and start a new box

# print("mesh_data.dat generated successfully!")

In [ ]:
T_err_cells = deepcopy(err_cells)
for level, cells in err_cells.items():
    Temp_cells = hs_dual.refine(cells, level, refine_T_neighbours=True, m=m)
    T_err_cells.update(my_dict_update(Temp_cells, T_err_cells))
#hs_dual.hmesh.plot_cells()

In [ ]:
for level, cells in T_err_cells.items():
    _ = hs.refine(cells, level, refine_T_neighbours=True, m=m)
hs.hmesh.plot_cells()

In [ ]:
def mapping_to_trapezoid(uv_points):
    original_shape = uv_points.shape
    uv_flat = uv_points.reshape(-1, 2)
    xy_flat = np.zeros_like(uv_flat)

    def bilinear(u_loc, v_loc, C00, C10, C01, C11):
        """Standard isoparametric Q1 interpolation"""
        return ((1-u_loc)*(1-v_loc)*C00 + 
                u_loc*(1-v_loc)*C10 + 
                (1-u_loc)*v_loc*C01 + 
                u_loc*v_loc*C11)
    
    P00 = np.array([0.0, 0.0])
    P10 = np.array([4.8, 4.4])
    P01 = np.array([0.0, 4.4])
    P11 = np.array([4.8, 6.0])

    xy_flat[:, 0] = bilinear(u_loc=uv_flat[:, 0], v_loc=uv_flat[:, 1],
                             C00=P00[0], C10=P10[0], C01=P01[0], C11=P11[0])
    xy_flat[:, 1] = bilinear(u_loc=uv_flat[:, 0], v_loc=uv_flat[:, 1],
                             C00=P00[1], C10=P10[1], C01=P01[1], C11=P11[1])
    
    
    return xy_flat.reshape(original_shape)

disconnected_mesh, thb_operators, N_max, trapez_midpoints = build_mesh(hs=hs, mapping=mapping_to_trapezoid)

In [ ]:
# topology, cell_types, geometry = dolfinx.plot.vtk_mesh(disconnected_mesh)
# grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# plotter = pyvista.Plotter()
# plotter.add_mesh(grid.shrink(.95), show_edges=False, color="#03bb85")
# plotter.view_xy()
# plotter.show(jupyter_backend="static")
# print(f"Number of points in PyVista grid: {grid.n_points}")

In [ ]:
tdim = disconnected_mesh.topology.dim
fdim = tdim - 1
disconnected_mesh.topology.create_connectivity(fdim, tdim)

coords = disconnected_mesh.geometry.x
# midpoints = np.mean(coords.reshape(-1, 4, 2), axis=1)
x_min = np.min(coords[:, 0])  # Find the leftmost x-value dynamically
x_max = np.max(coords[:, 0])
y_min = np.min(coords[:, 1])
y_max = np.max(coords[:, 1])
y_min2 = np.min(coords[:, 1][np.isclose(coords[:, 0], x_max)])
y_max2 = np.max(coords[:, 1][np.isclose(coords[:, 0], x_min)])

T00 = np.array([x_min, y_min], dtype=float)
T01 = np.array([x_max, y_min2], dtype=float)
T10 = np.array([x_min, y_max2], dtype=float)
T11 = np.array([x_max, y_max], dtype=float)

def left_boundary(x):
    return np.isclose(x[0], x_min)
def right_boundary(x):
    return np.isclose(x[0], x_max)
def bottom_boundary(x):
    return np.isclose(x[1], x[0]*T01[1]/T01[0])
def top_boundary(x):
    return np.isclose(x[1], T10[1]+x[0]*(T11[1]-T10[1])/T11[0])
def bottom_right_top_boundary(x):
    return right_boundary(x)|bottom_boundary(x)|top_boundary(x)
def not_right(x):
    return ~right_boundary(x)


In [ ]:
left_facets = dolfinx.mesh.locate_entities_boundary(
    disconnected_mesh, 
    fdim, 
    left_boundary
)

right_facets = dolfinx.mesh.locate_entities_boundary(
    disconnected_mesh, 
    fdim, 
    right_boundary
)

#sorted_facets = np.sort(right_facets)

# Assign a marker ID (e.g., 1) to these facets
facet_values = np.full_like(right_facets, 1, dtype=np.int32)

# Create the MeshTags object
facet_tags = dolfinx.mesh.meshtags(
    disconnected_mesh, fdim, right_facets, facet_values
)

dim = disconnected_mesh.topology.dim
legendre_elt = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0,
    shape=(dim,),
    lagrange_variant=basix.LagrangeVariant.legendre
)
V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
# print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")

#boundary_facets = dolfinx.mesh.locate_entities_boundary(disconnected_mesh, facet_dim, Gamma_N)
custom_metadata = {"quadrature_degree": 6}
ds_custom = ufl.Measure("ds", domain=disconnected_mesh, subdomain_data=facet_tags, metadata=custom_metadata)
dx_custom = ufl.Measure("dx", domain=disconnected_mesh, metadata=custom_metadata)

E=70.0
nu = 1./3.
lmbda = E*nu/(1.+nu)/(1.-2.*nu)
#mu = E / 2. / (1. + nu)
mu = E/2./(1.+nu)
lmbda_c = dolfinx.fem.Constant(disconnected_mesh, lmbda)
mu_c = dolfinx.fem.Constant(disconnected_mesh, mu)

def epsilon(v):
    return ufl.sym(ufl.grad(v))

def sigma(v):
    return lmbda_c * ufl.tr(epsilon(v)) * ufl.Identity(dim) + 2. * mu_c * epsilon(v)


u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
T = dolfinx.fem.Constant(
    disconnected_mesh, 
    dolfinx.default_scalar_type((0., 6.25))
)
a = ufl.inner(sigma(u),epsilon(v))*dx_custom
L_facet = ufl.inner(T, v)*ds_custom(1)

msh = disconnected_mesh

In [ ]:
dofmap, padded_cells_to_dofs = build_dofmap(hierarchical_space=hs, mesh=disconnected_mesh, N_max=N_max, morton=False)
custom_mapping = FastMidpointMapper(hs, trapez_midpoints)
C_func, C_space = fill_function_space(hierachical_space=hs, mesh=disconnected_mesh, N_max = N_max, 
                                      thb_operators=thb_operators, mapping_function=custom_mapping)
V_spline = create_spline_space(cells_to_dofs=padded_cells_to_dofs, mesh=disconnected_mesh,
                               N_max=N_max, mult_factor=2)

In [ ]:
local_dofs = (hs.degrees[0]+1)**2
tabulate_A = make_vector_bilinear_kernel(disconnected_mesh, a, padded_dofs=N_max, local_dofs=local_dofs)
tabulate_L_facet = make_vector_linear_kernel(disconnected_mesh, L_facet, padded_dofs=N_max, local_dofs=local_dofs)

In [ ]:
facet_dim = msh.topology.dim-1

boundary_facets = dolfinx.mesh.locate_entities_boundary(msh, facet_dim, right_boundary)
msh.topology.create_connectivity(facet_dim, msh.topology.dim)
msh.topology.create_connectivity(msh.topology.dim, facet_dim)

# dictionary of facet->cell
f_to_c = msh.topology.connectivity(facet_dim, msh.topology.dim)
# dictionary of cell->array[facets]
c_to_f = msh.topology.connectivity(msh.topology.dim, facet_dim)

boundary_entities = []
# Loop over all edges that belong to our exterior
for f in boundary_facets:
    # returns the cells linked to this edge
    cells = f_to_c.links(f)
    # Exterior facets only have 1 attached cell, 
    # hence get the first one 
    c = cells[0] 
    
    # Find the local index (e.g., 0, 1, 2, or 3 for quads) of facet f within cell c
    local_f = np.where(c_to_f.links(c) == f)[0][0]
    #print(f"f={f}, cell={c}, local_f = {local_f}")
    
    boundary_entities.extend([c, local_f])

# FEniCSx custom form arrays must be typed as int32
boundary_entities = np.array(boundary_entities, dtype=np.int32)

In [ ]:
lhs_constants = ufl.algorithms.analysis.extract_constants(a)
# Extract the underlying C++ objects in that exact order
cpp_constants_lhs = [c._cpp_object for c in lhs_constants]

rhs_constants = ufl.algorithms.analysis.extract_constants(L_facet)
cpp_constants_rhs = [c._cpp_object for c in rhs_constants]

formtype = dolfinx.fem.form_cpp_class(dtype)
cells = np.arange(msh.topology.index_map(msh.topology.dim).size_local, dtype=np.int32)

integrals = {dolfinx.fem.IntegralType.cell: [
    (0, tabulate_A.address, cells, np.array([0], dtype=np.int8))]}

a_cond = dolfinx.fem.Form( 
    formtype( 
        spaces=[V_spline._cpp_object, V_spline._cpp_object],
        integrals=integrals,
        coefficients=[C_func._cpp_object],
              constants=cpp_constants_lhs,
              need_permutation_data=False,
              entity_maps=[], 
              mesh=msh._cpp_object)
)

integrals_rhs = {dolfinx.fem.IntegralType.exterior_facet: [(0, tabulate_L_facet.address, boundary_entities, np.array([0], dtype=np.int8))]}
l_cond = dolfinx.fem.Form(
    formtype(
        spaces=[V_spline._cpp_object],
        integrals=integrals_rhs, 
        coefficients=[C_func._cpp_object], 
        constants=cpp_constants_rhs, 
        need_permutation_data=False, 
        entity_maps=[], 
        mesh=msh._cpp_object
    )
)

In [ ]:
#forbidden_indices = get_spline_indices_left(hs, dofmap)
forbidden_indices = enforce_dirichlet_boundary(hs, dofmap, left=True)
if forbidden_indices is not None and len(forbidden_indices) > 0:
    forbidden_indices_vec = np.empty(2 * len(forbidden_indices), dtype=np.int32)
    forbidden_indices_vec[0::2] = 2 * forbidden_indices      # X DOFs
    forbidden_indices_vec[1::2] = 2 * forbidden_indices + 1  # Y DOFs
    forbidden_indices = forbidden_indices_vec

x_vec = solve_problem_vector_field(hs=hs, a = a_cond, lhs=l_cond, dirichlet_indices=forbidden_indices, 
                                   dummy_index=np.max(padded_cells_to_dofs), V_spline=V_spline)
u_dg = map_spline_to_legendre(hs=hs, V=V, C_func=C_func, N_max=N_max, mesh=disconnected_mesh, 
                                           cells_to_dofs=padded_cells_to_dofs, u_sol=x_vec, vector_field=True)

In [ ]:
print(f"Top right corner vertical displacement:\n ({int(x_vec.shape[0]/2)}, {np.max(u_dg.x.array.reshape(-1, 2)[:,1])*10:.6f})\nmm")

In [ ]:
import pyvista
import numpy as np
import dolfinx
from dolfinx.plot import vtk_mesh

dim = msh.geometry.dim

# 1. Create a plot-friendly function space: Continuous Galerkin (Lagrange) degree 1
# This guarantees that DOFs match the physical vertices of the mesh.
V_plot = dolfinx.fem.functionspace(msh, ("Lagrange", 2, (dim,)))

# 2. Interpolate your Legendre/DG function into this nodal space
u_plot = dolfinx.fem.Function(V_plot)
u_plot.interpolate(u_dg)

# 3. Generate the VTK mesh from the plotting space
topology, cell_types, geometry = vtk_mesh(V_plot)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# 4. Extract the solution array and reshape it to N x dim
# Because V_plot is a vector space, DOFs are interleaved (x0, y0, x1, y1...)
u_values = u_plot.x.array.reshape(-1, dim)

# 5. PyVista requires 3D vectors to warp the mesh. Pad the 2D vectors with Z=0.
u_3d = np.zeros((u_values.shape[0], 3), dtype=np.float64)
u_3d[:, :dim] = u_values

# 6. Attach the 3D displacement vectors to the grid
grid.point_data["Displacement"] = u_3d
grid.set_active_vectors("Displacement")

# Optional: Calculate displacement magnitude to use for coloring
grid.point_data["Displacement_Magnitude"] = np.linalg.norm(u_3d, axis=1)

# 7. Warp the grid by the displacement
warp_factor = 1.0
warped_grid = grid.warp_by_vector("Displacement", factor=warp_factor)

# 8. Plotting
# plotter = pyvista.Plotter()
# plotter.add_text(f"Deformed Mesh (magnified {warp_factor}x)", font_size=14)

# # Show the original undeformed mesh as a wireframe
# plotter.add_mesh(grid, style="wireframe", color="black", opacity=0.3, label="Undeformed")

# # Show the deformed mesh, coloring it by the magnitude of the displacement
# plotter.add_mesh(
#     warped_grid, 
#     show_edges=False, 
#     scalars="Displacement_Magnitude", # Color by magnitude instead of the vector array
#     cmap="coolwarm", 
#     label="Deformed"
# )

# plotter.view_xy()  # Set camera to view the X-Y plane directly
# plotter.show(jupyter_backend="static")

In [ ]:
# ==============================================================================
# FIX 1: Manually build a clean, non-triangulated quad grid for the undeformed mesh
# ==============================================================================
num_cells = msh.topology.index_map(msh.topology.dim).size_local
geom_dofmap = msh.geometry.dofmap

# Extract just the local cells and ensure we only take the 4 corner nodes 
# (in case your mesh geometry is higher-order)
quad_nodes = geom_dofmap[:num_cells, :4]

# Create a column of 4s to tell PyVista that each cell is a 4-node quad
padding = np.full((num_cells, 1), 4, dtype=np.int32)

# Glue them together: [4, v0, v1, v2, v3, 4, v0, v1, ...] and flatten
cells_quad = np.hstack([padding, quad_nodes]).flatten()

# Define cell types and build the clean wireframe grid
cell_types_quad = np.full(num_cells, pyvista.CellType.QUAD, dtype=np.uint8)
grid_undeformed_clean = pyvista.UnstructuredGrid(cells_quad, cell_types_quad, msh.geometry.x)

# ==============================================================================
# FIX 2: Extract smooth, curved macro-boundaries from the high-order warped grid
# ==============================================================================
# Internal sub-triangles are perfectly flat (0° angle). Element boundaries have a 
# tiny physical kink due to FE approximation. feature_angle catches exactly those kinks!
deformed_macro_edges = warped_grid.extract_feature_edges(
    feature_angle=0.5,        # Adjust slightly (e.g., between 0.1 and 2.0) if needed
    boundary_edges=True,       # Keeps the outer boundary of the domain
    non_manifold_edges=False,
    manifold_edges=False
)


# ==============================================================================
# 8. Plotting Everything Together
# ==============================================================================
# plotter = pyvista.Plotter(window_size=[2500, 2500])
#plotter.add_text(f"Deformed Mesh (magnified {warp_factor}x)", font_size=14)

# Show the clean original macro mesh as a wireframe (Guaranteed NO diagonals)
# plotter.add_mesh(
#     grid_undeformed_clean, 
#     style="wireframe", 
#     color="black", 
#     opacity=0.2, 
#     label="Undeformed"
# )

# Show the deformed mesh (Smooth high-order surface coloring, no edges)
# plotter.add_mesh(
#     warped_grid, 
#     show_edges=False, 
#     scalars="Displacement_Magnitude", 
#     cmap="coolwarm", 
#     label="Deformed",
#     show_scalar_bar=True
# )

# # Overlay the perfectly smooth, curved macro edges (No diagonals, no clipping!)
# plotter.add_mesh(
#     deformed_macro_edges, 
#     color="black", 
#     line_width=2, 
#     label="Deformed Elements",
# )

#plotter.view_xy()  # Set camera to view the X-Y plane directly
#plotter.show(jupyter_backend="static")
#plotter.screenshot("cooks_membrane.png")

# Dual

In [ ]:
legendre_elt_dual = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0_dual,
    shape=(dim,),
    lagrange_variant=basix.LagrangeVariant.legendre
)
V_dual = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt_dual)

z,v_dual = ufl.TrialFunction(V_dual), ufl.TestFunction(V_dual)

x = ufl.SpatialCoordinate(disconnected_mesh)
# constant_zero = dolfinx.fem.Constant(disconnected_mesh, 0.)
s = 0.075
def mollifier(rho):
    return np.exp(-1./(1.-rho**2))*rho
integral, _ = quad(mollifier, 0., 1.)
integral_c = dolfinx.fem.Constant(disconnected_mesh, 2.*np.pi*s**2*integral)
r2 = (x[0]-x_max)**2+(x[1]-y_max)**2
mollifier_inside = ufl.exp(-1./(1.-(r2/(s**2) ) ))
rhs_dual = ufl.as_vector((0., ufl.conditional(
                        condition=ufl.lt(left=r2, right=s**2), 
                        true_value=mollifier_inside / integral_c, 
                        false_value=0.)
))
a_dual = ufl.inner(sigma(v_dual), epsilon(z)) * dx_custom
L_cell_dual = ufl.inner(rhs_dual, v_dual) * dx_custom 

In [ ]:
_, thb_operators_dual, N_max_dual, trapez_midpoints_dual = build_mesh(hs=hs_dual, mapping=mapping_to_trapezoid)

custom_mapping_dual = FastMidpointMapper(hs_dual, trapez_midpoints_dual)

dof_map_dual, padded_cells_to_dofs_dual = build_dofmap(hierarchical_space=hs_dual, mesh=disconnected_mesh, N_max=N_max_dual, morton=False)

C_func_dual, C_space_dual = fill_function_space(hierachical_space=hs_dual, mesh=disconnected_mesh, N_max=N_max_dual, 
                                                thb_operators=thb_operators_dual, mapping_function=custom_mapping_dual)

V_spline_dual = create_spline_space(cells_to_dofs=padded_cells_to_dofs_dual, mesh=disconnected_mesh,
                               N_max=N_max_dual, mult_factor=2)

local_dofs_dual = (p0_dual+1)**2
tabulate_A_dual = make_vector_bilinear_kernel(disconnected_mesh, a_dual, padded_dofs=N_max_dual, local_dofs=local_dofs_dual)
tabulate_L_cell_dual = make_vector_linear_kernel(disconnected_mesh, L_cell_dual, padded_dofs=N_max_dual, local_dofs=local_dofs_dual)

In [ ]:
lhs_constants_dual = ufl.algorithms.analysis.extract_constants(a_dual)
# Extract the underlying C++ objects in that exact order
cpp_constants_lhs_dual = [c._cpp_object for c in lhs_constants_dual]

rhs_constants_dual = ufl.algorithms.analysis.extract_constants(L_cell_dual)
cpp_constants_rhs_dual = [c._cpp_object for c in rhs_constants_dual]

integrals_dual = {dolfinx.fem.IntegralType.cell: [
    (0, tabulate_A_dual.address, cells, np.array([0], dtype=np.int8))]}

a_cond_dual = dolfinx.fem.Form( 
    formtype( 
        spaces=[V_spline_dual._cpp_object, V_spline_dual._cpp_object],
        integrals=integrals_dual,
        coefficients=[C_func_dual._cpp_object],
              constants=cpp_constants_lhs_dual,
              need_permutation_data=False,
              entity_maps=[], 
              mesh=msh._cpp_object)
)

integrals_rhs_dual = {dolfinx.fem.IntegralType.cell: [(0, tabulate_L_cell_dual.address, cells, np.array([0], dtype=np.int8))]}
l_cond_dual = dolfinx.fem.Form(
    formtype(
        spaces=[V_spline_dual._cpp_object],
        integrals=integrals_rhs_dual, 
        coefficients=[C_func_dual._cpp_object], 
        constants=cpp_constants_rhs_dual, 
        need_permutation_data=False, 
        entity_maps=[], 
        mesh=msh._cpp_object
    )
)

In [ ]:
#forbidden_indices_dual = get_spline_indices_left(hs_dual, dof_map_dual)
forbidden_indices_dual = enforce_dirichlet_boundary(hs_dual, dof_map_dual, left=True)
if forbidden_indices_dual is not None and len(forbidden_indices_dual) > 0:
    forbidden_indices_vec_dual = np.empty(2 * len(forbidden_indices_dual), dtype=np.int32)
    forbidden_indices_vec_dual[0::2] = 2 * forbidden_indices_dual      # X DOFs
    forbidden_indices_vec_dual[1::2] = 2 * forbidden_indices_dual + 1  # Y DOFs
    forbidden_indices_dual = forbidden_indices_vec_dual

In [ ]:
z_vec = solve_problem_vector_field(hs=hs_dual, a = a_cond_dual, lhs=l_cond_dual, dirichlet_indices=forbidden_indices_dual, 
                                   dummy_index=np.max(padded_cells_to_dofs_dual), V_spline=V_spline_dual)
z_dg = map_spline_to_legendre(hs=hs_dual, V=V_dual, C_func=C_func_dual,
                                           N_max=N_max_dual, mesh=disconnected_mesh, 
                                           cells_to_dofs=padded_cells_to_dofs_dual, u_sol=z_vec, vector_field=True)

Error residuals estimates

In [ ]:
DG0 = dolfinx.fem.functionspace(disconnected_mesh, ("DG", 0))
v_dg = ufl.TestFunction(DG0)

i_h_z_h = dolfinx.fem.Function(V)
i_h_z_h.interpolate(z_dg)
dual_weight = z_dg - i_h_z_h

R_expr = ufl.div(sigma(u_dg))
cell_form = ufl.inner(R_expr, dual_weight)*v_dg*dx_custom

n_normal = ufl.FacetNormal(disconnected_mesh)
flux_jump = ufl.jump(sigma(u_dg), n_normal)
r_val = 0.5 * flux_jump
custom_dS = ufl.Measure("dS", domain=disconnected_mesh, metadata=custom_metadata)



facet_form_dS = (
    ufl.inner(r_val, dual_weight('+')) * v_dg('+')
    + ufl.inner(r_val, dual_weight('-')) * v_dg('-')
) * custom_dS 

traction_computed = ufl.dot(sigma(u_dg), n_normal)
facet_form_ds = ufl.inner(dual_weight,T-traction_computed)*v_dg*ds_custom(1)

eta_form = cell_form+ facet_form_dS + facet_form_ds
eta_vec = dolfinx.fem.assemble_vector(dolfinx.fem.form(eta_form))
err_cells = dorfler_marking(hs, 0.6, dolfinx.fem.form(eta_form))
#eta_vec.scatter_forward()
#eta_K = np.abs(eta_vec.array)